# 02 — Traffic Allocation & Route Generation

Generates `.rou.xml` and `.sumocfg` files for each ward based on:
- Zone type (commercial, residential, etc.)
- Scenario configuration (normal, peak, chaos, etc.)
- Vehicle profiles (7 vehicle types)
- Boundary edges from `boundaries.json`

**Prerequisites:** Run `01_osm_preprocessing.ipynb` first.

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 1. Select Wards and Scenario

In [ ]:
import json
from configs.scenarios import SCENARIOS, list_scenarios
from configs.vehicle_profiles import VEHICLE_PROFILES
from configs.traffic_profiles import ZONE_PROFILES

# Show available scenarios
print("Available scenarios:")
for sid in list_scenarios():
    desc = SCENARIOS[sid]["description"]
    print(f"  • {sid}: {desc}")

# Select scenario
SCENARIO_ID = "normal"  # Change as needed
print(f"\nSelected: {SCENARIO_ID}")

In [ ]:
# Find processed wards
processed_dir = PROJECT_ROOT / "maps" / "processed"
ward_dirs = sorted(d.name for d in processed_dir.iterdir() if d.is_dir() and (d / "ward.net.xml").exists())
print(f"Processed wards ready for route generation: {len(ward_dirs)}")
for w in ward_dirs:
    print(f"  • {w}")

## 2. Generate Routes for Each Ward

In [ ]:
from src.traffic_generator import TrafficGenerator

route_results = []

for ward_id in ward_dirs:
    print(f"\nGenerating routes for {ward_id}...")
    gen = TrafficGenerator(PROJECT_ROOT, ward_id, scenario_id=SCENARIO_ID)
    
    route_path = gen.generate_ward_routes()
    cfg_path = gen.generate_ward_sumocfg()
    
    route_results.append({
        "ward_id": ward_id,
        "route_file": str(route_path),
        "sumocfg": str(cfg_path),
        "route_size_kb": route_path.stat().st_size / 1024,
    })
    print(f"  ✅ {route_path.name} ({route_path.stat().st_size / 1024:.1f} KB)")
    print(f"  ✅ {cfg_path.name}")

## 3. Verify: Vehicle Mix Distribution

In [ ]:
import matplotlib.pyplot as plt

scenario = SCENARIOS[SCENARIO_ID]
mix = scenario["vehicle_mix"]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#6C7A89', '#E74C3C', '#5DADE2', '#27AE60', '#8D6E63', '#FFFFFF', '#2C3E50']
ax.barh(list(mix.keys()), list(mix.values()), color=colors[:len(mix)])
ax.set_xlabel("Proportion")
ax.set_title(f"Vehicle Mix — Scenario: {SCENARIO_ID}")
plt.tight_layout()
plt.show()

## 4. Quick Test: Load a Ward in SUMO

In [ ]:
if route_results:
    test = route_results[0]
    print(f"To test {test['ward_id']} in SUMO GUI:")
    print(f'  sumo-gui -c "{test["sumocfg"]}"')
else:
    print("No routes generated yet.")